In [ ]:
# launch the offline engine
import asyncio
import io
from multiprocessing import freeze_support
import os

from PIL import Image
import requests
import sglang as sgl

import sglang.global_env as global_env
from sglang.srt.conversation import chat_templates
from sglang.test.test_utils import is_in_ci
from sglang.utils import async_stream_and_merge, stream_and_merge

if is_in_ci():
    import patch


In [ ]:
import torch

MIN_MEMORY  = 1 << 20 # 1MB as the minimum memory
MAX_MEMORY  = 1 << 40 # 1TB as the maximum memory

lists = []

num_gpus = torch.cuda.device_count()
which = 0

mem_list = [1 << 30] * num_gpus

while True:
    current = mem_list[which]
    while True:
        assert isinstance(current, int)
        if current < MIN_MEMORY:
            current = MIN_MEMORY
        if current > MAX_MEMORY:
            current = MAX_MEMORY
        try:
            a = torch.zeros(current, dtype=torch.bool, device="cuda:0")
            lists.append(a)
            current = current * 2
            print(f"Allocated {torch.cuda.memory_allocated() / 1e9} GB")
        except torch.OutOfMemoryError:
            current = current // 2
            break
    mem_list[which] = current
    which = (which + 1) % num_gpus

In [10]:
import torch
import os

FILE_1 = [ f"/sgl-workspace/sglang/logs/2025-03-26_06-40-54/input_{i}.pt" for i in range(1000) ]
FILE_2 = [ f"/sgl-workspace/sglang/logs/2025-03-26_06-42-59/input_{i}.pt" for i in range(1000) ]

token_layer_to_experts = {}

for f in FILE_2:
    if not os.path.exists(f):
        continue
    with open(f, "rb") as f:
        database = torch.load(f)
        input_ids = database["input_ids"]
        input_len = input_ids.shape[0]
        
        for i in range(input_len):
            token = input_ids[i].item()
            for idx in database:
                activation = database[idx]
                if idx == "input_ids":
                    continue
                if (token, idx) not in token_layer_to_experts:
                    token_layer_to_experts[(token, idx)] = [torch.topk(activation[i], 6)]
                else:
                    token_layer_to_experts[(token, idx)].append(torch.topk(activation[i], 6))


/tmp/ipykernel_384811/2052078650.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  database = torch.load(f)


In [12]:

with open(FILE_1[5], "rb") as f:
    input_0 = torch.load(f)
    print(input_0["input_ids"].shape)
    print(input_0["input_ids"])
    print(input_0[8].shape)
    print(torch.topk(input_0[26], 5))

with open(FILE_1[13], "rb") as f:
    input_0 = torch.load(f)
    print(input_0["input_ids"].shape)
    print(input_0["input_ids"])
    print(input_0[8].shape)
    print(torch.topk(input_0[26], 5))


torch.Size([1])
tensor([608], device='cuda:0')
torch.Size([1, 64])
torch.return_types.topk(
values=tensor([[2.5625, 2.4219, 1.3984, 1.0703, 1.0156]], device='cuda:0',
       dtype=torch.bfloat16),
indices=tensor([[ 2,  4, 50, 54, 30]], device='cuda:0'))
torch.Size([1])
tensor([608], device='cuda:0')
torch.Size([1, 64])
torch.return_types.topk(
values=tensor([[2.5312, 2.5312, 1.4219, 1.0781, 1.0547]], device='cuda:0',
       dtype=torch.bfloat16),
indices=tensor([[ 4,  2, 50, 54, 46]], device='cuda:0'))


/tmp/ipykernel_384811/3236507500.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  input_0 = torch.load(f)
/tmp/ipykernel_384811/3236507500.py:9: FutureWarning: You are us

In [20]:
for token, idx in token_layer_to_experts:
    if len(token_layer_to_experts[(token, idx)]) < 2 or idx < 10:
        continue
    print(token, idx)
    for expert_list in token_layer_to_experts[(token, idx)]:
        top_k = expert_list.indices.tolist()
        top_k.sort()
        print(top_k)


245 10
[4, 27, 33, 38, 59, 61]
[4, 26, 33, 36, 38, 61]
[6, 19, 31, 59, 61, 62]
[3, 6, 31, 59, 61, 62]
[3, 6, 31, 50, 59, 61]
[3, 6, 50, 51, 59, 61]
[3, 6, 50, 51, 59, 61]
[3, 6, 31, 50, 59, 61]
[3, 6, 31, 50, 59, 61]
[3, 6, 31, 50, 59, 61]
[6, 19, 31, 50, 59, 61]
[3, 6, 31, 59, 61, 62]
[6, 19, 31, 50, 59, 61]
[6, 26, 39, 50, 59, 61]
[4, 31, 33, 38, 41, 61]
[4, 33, 38, 41, 50, 61]
245 11
[16, 20, 25, 30, 43, 54]
[19, 39, 45, 50, 54, 62]
[5, 10, 20, 25, 37, 63]
[1, 5, 7, 10, 25, 37]
[5, 7, 10, 25, 37, 63]
[5, 20, 25, 37, 53, 63]
[5, 10, 20, 25, 43, 63]
[5, 10, 20, 25, 43, 63]
[5, 10, 20, 25, 43, 63]
[5, 10, 20, 25, 43, 63]
[1, 5, 10, 25, 37, 63]
[1, 5, 10, 25, 37, 63]
[1, 5, 10, 25, 26, 63]
[1, 5, 20, 39, 45, 63]
[20, 30, 39, 45, 51, 62]
[2, 20, 30, 45, 51, 53]
245 12
[30, 34, 41, 44, 47, 53]
[20, 30, 43, 44, 49, 53]
[9, 10, 13, 19, 53, 57]
[9, 13, 19, 41, 53, 57]
[9, 13, 19, 41, 53, 57]
[19, 28, 41, 45, 53, 57]
[19, 28, 41, 47, 53, 57]
[19, 28, 41, 47, 53, 57]
[37, 41, 47, 53, 56, 57]
[